# Распознавание аудио из датасета через GigaAM (GitHub)

Ноутбук:
- читает CSV датасета;
- берёт путь к аудио **только** из `absolute_path`;
- загружает GigaAM **напрямую из GitHub**;
- распознаёт текст;
- сохраняет word-level timestamps;
- пишет итоговый файл со всеми исходными метками и новыми колонками результата.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/salute-developers/GigaAM.git"
REPO_DIR = Path("GigaAM")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%pip install -q -e ./GigaAM

# Если в датасете есть длинные аудио и нужен transcribe_longform,
# раскомментируйте строку ниже и перезапустите kernel.
# %pip install -q -e "./GigaAM[longform]"


In [ ]:
import os
import json
from pathlib import Path, PureWindowsPath

import pandas as pd
import torch
from tqdm.auto import tqdm

import gigaam

In [ ]:
DATASET_PATH = Path(
    "D:\\project\\multicriteria-dialog-audit-ml\\node\\Выгрузка_updated.csv"
)
# MODEL_NAME = "v3_e2e_rnnt"
MODEL_NAME = "v3_e2e_ctc"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Включайте только если установлены longform-зависимости.
ENABLE_LONGFORM = True

# Нужен только для longform через pyannote.
HF_TOKEN = os.environ.get("HF_TOKEN", "")

print(f"Using device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Enable longform: {ENABLE_LONGFORM}")
print(f"HF Token provided: {'Yes' if HF_TOKEN else 'No'}")

df = pd.read_csv(DATASET_PATH)
df.head()

In [ ]:
model = gigaam.load_model(MODEL_NAME, device=DEVICE)
model

In [ ]:
def normalize_dataset_path(path_value: str) -> Path:
    path_str = str(path_value).strip()
    if not path_str:
        raise ValueError("Пустой путь к аудио")
    return Path(PureWindowsPath(path_str))


def words_to_list(words):
    print(words)
    if not words:
        return None
    return [
        {
            "text": w.text,
            "start": round(float(w.start), 3),
            "end": round(float(w.end), 3),
        }
        for w in words
    ]


def segments_to_list(segments):
    if not segments:
        return None
    return [
        {
            "text": s.text,
            "start": round(float(s.start), 3),
            "end": round(float(s.end), 3),
            "words": words_to_list(s.words),
        }
        for s in segments
    ]


def transcribe_one_file(audio_path_raw: str) -> dict:
    audio_path = normalize_dataset_path(audio_path_raw)

    if not audio_path.exists():
        raise FileNotFoundError(f"Файл не найден: {audio_path}")

    try:
        result = model.transcribe(str(audio_path), word_timestamps=True)
        return {
            "predicted_transcription": result.text,
            "word_timestamps_json": json.dumps(
                words_to_list(result.words), ensure_ascii=False
            ),
            "segment_timestamps_json": None,
            "status": "ok",
            "error_message": None,
        }

    except ValueError as e:
        if "Too long wav file" not in str(e):
            raise
        if not ENABLE_LONGFORM:
            raise RuntimeError(
                "Аудио длиннее 25 секунд. Установите longform-зависимости "
                "и включите ENABLE_LONGFORM = True."
            ) from e

        long_result = model.transcribe_longform(str(audio_path), word_timestamps=True)

        return {
            "predicted_transcription": long_result.text,
            "word_timestamps_json": json.dumps(
                words_to_list(long_result.words), ensure_ascii=False
            ),
            "segment_timestamps_json": json.dumps(
                segments_to_list(long_result.segments),
                ensure_ascii=False,
            ),
            "status": "ok_longform",
            "error_message": None,
        }

In [ ]:
tqdm.pandas()

results = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        pred = transcribe_one_file(row["absolute_path"])
    except Exception as e:
        pred = {
            "predicted_transcription": None,
            "word_timestamps_json": None,
            "segment_timestamps_json": None,
            "status": "error",
            "error_message": str(e),
        }
    results.append(pred)

pred_df = pd.DataFrame(results)
result_df = pd.concat([df.reset_index(drop=True), pred_df], axis=1)
result_df.head()

In [ ]:
OUTPUT_PARQUET = Path("recognized_calls_gigaam_" + MODEL_NAME + "_github.parquet")

result_df.to_parquet(OUTPUT_PARQUET, index=False)

print(OUTPUT_PARQUET.resolve())

In [ ]:
result_df["status"].value_counts(dropna=False)